# Qwen 2.5-7B on the 5090: fp8 against bf16 through the full pipeline

Ten cells x 3 passes x 2 query halves = **60 lanes per precision**, in four waves (18, 18, 12, 12); fp8 first, then bf16.
Run cells 1 to 3 in order and **keep cell 3 running**: it shows progress, switches the server from fp8 to bf16
and builds the zips. `qprec5090_fp8_results.zip` appears as soon as the fp8 half is done, `qprec5090_results.zip`
at the end. Re-running cell 3 is safe: finished lanes are skipped.

In [1]:
# 1. Setup
import os, sys, subprocess, time, json, csv, io, zipfile, shutil, glob, urllib.request
csv.field_size_limit(10**8)
def _find_repo():
    # The unpacked harness: ~/work/skyrag, or the one folder under ~/work holding it (e.g. a project folder).
    base = os.path.expanduser("~/work")
    hits = [os.path.dirname(p) for p in glob.glob(os.path.join(base, "skyrag", "BUNDLE.json"))
            + glob.glob(os.path.join(base, "*", "skyrag", "BUNDLE.json"))]
    if len(hits) != 1:
        raise SystemExit(f"expected exactly one unpacked harness under {base}, found {hits or 'none'}")
    return hits[0]
REPO = _find_repo(); WORK = os.path.dirname(REPO); OUT = os.path.join(REPO, "outputs")
print("harness:", REPO)
os.makedirs(OUT, exist_ok=True)
print("bundle commit:", json.load(open(os.path.join(REPO, "BUNDLE.json")))["commit"][:7])
VLLM = shutil.which("vllm") or "/opt/conda/bin/vllm"
PY = os.path.join(os.path.dirname(VLLM), "python") if os.path.exists(VLLM) else sys.executable

# Two query halves of the benchmark, written once (the same files Run A used).
src = [l for l in open(os.path.join(REPO, "benchmarks/queries_v1.jsonl"), encoding="utf-8").read().splitlines() if l.strip()]
HALVES = {1: "benchmarks/queries_v1_h1.jsonl", 2: "benchmarks/queries_v1_h2.jsonl"}
for k, part in ((1, src[:25]), (2, src[25:])):
    p = os.path.join(REPO, HALVES[k])
    if not os.path.exists(p):
        open(p, "w", encoding="utf-8").write("\n".join(part) + "\n")

HF, SERVED, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct", "qwen/qwen-2.5-7b-instruct", "32768"
PRECISIONS = ["fp8", "bf16"]                      # fp8 first: it is the half that does not exist yet
CONTEXTS = {"s": [], "i": ["--candidate-injection", "--candidate-limit", "100"]}
CORE = [("a2", "2", None), ("a2_s1", "2", "schema"), ("a3", "3", None)]
EXTRA = [("a2_sw", "2", "schema_wording"), ("a3_p1", "3", "plain")]
WAVES = [("s", "core", CORE), ("i", "core", CORE), ("s", "decomposition", EXTRA), ("i", "decomposition", EXTRA)]
ROWS = 25; GPU_UTIL = "0.85"; PORT = 8000

def lane_name(prec, ctx, cell, rep, h):
    return f"qprec5090_qwen_{prec}_{ctx}_{cell}_r{rep}_h{h}"

from getpass import getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF read token: ").strip()
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)

harness: /home/jovyan/work/TA-Sean-Matthew/skyrag
bundle commit: b3153b8


HF read token:  ········


NVIDIA GeForce RTX 5090, 26829 MiB, 32607 MiB

               total        used        free      shared  buff/cache   available
Mem:             125           7          15           0         104         118
Swap:              7           0           7



In [2]:
# 2. Helpers
def healthy():
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def stop_any_server():
    subprocess.run(["pkill", "-f", "vllm serve"]); time.sleep(10)
    while healthy():
        time.sleep(5)

def start_server(precision):
    args = [VLLM, "serve", HF, "--served-model-name", SERVED, "--port", str(PORT), "--dtype", "bfloat16",
            "--max-model-len", MAX_LEN, "--gpu-memory-utilization", GPU_UTIL, "--generation-config", "vllm"]
    if precision == "fp8":
        args += ["--quantization", "fp8"]
    log = open(os.path.join(WORK, f"qprec5090_server_qwen_{precision}.log"), "w")
    p = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while not healthy():
        if p.poll() is not None:
            raise SystemExit(f"Server failed; paste the end of qprec5090_server_qwen_{precision}.log to Claude.")
        time.sleep(5)
    print(f"  {precision} server ready after {(time.time() - t0) / 60:.1f} min", flush=True)

ENV = {**os.environ, "OPENROUTER_BASE_URL": f"http://localhost:{PORT}/v1", "OPENROUTER_API_KEY": "local-vllm",
       "SKYRAG_OPENROUTER_PROVIDER": "local-vllm-rtx5090", "SKYRAG_DEVICE": "cpu",
       "PYTHONPATH": os.path.join(REPO, "src"), "OMP_NUM_THREADS": "2", "TOKENIZERS_PARALLELISM": "false"}

def launch(name, args):
    out = os.path.join(OUT, name)
    log = open(os.path.join(OUT, f"lane_{name}.log"), "w")
    return subprocess.Popen([PY, "-m", "skyrag.cli", "eval", *args, "-o", out + "/"], cwd=REPO, env=ENV,
                            stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

def rows_of(name):
    try:
        return list(csv.DictReader(io.open(os.path.join(OUT, name, "benchmark_eval.csv"), encoding="utf-8")))
    except Exception:
        return []

def status(name):
    rows = rows_of(name)
    try:
        log = open(os.path.join(OUT, f"lane_{name}.log"), errors="replace").read()
    except Exception:
        log = ""
    flag = ""
    if "circuit breaker" in log.lower(): flag = "  CIRCUIT BREAKER"
    elif "NOT VALID AS RESULTS" in log: flag = "  EMBEDDER FALLBACK"
    elif "Traceback" in log: flag = "  TRACEBACK (see log)"
    failed = sum(r.get("status") == "cell_failed" for r in rows)
    k = int(10 * len(rows) / ROWS)
    return len(rows), f"  {name[len('qprec5090_qwen_'):]:22s} [{'#' * k}{'.' * (10 - k)}] {len(rows):2d}/{ROWS} failed={failed}{flag}"

def zip_run(zname, pattern):
    path = os.path.join(WORK, zname)
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(os.path.join(OUT, pattern)) + glob.glob(os.path.join(WORK, "qprec5090_server_*.log")):
            if os.path.isdir(f):
                for x in os.listdir(f):
                    z.write(os.path.join(f, x), os.path.join("outputs", os.path.basename(f), x))
            else:
                z.write(f, os.path.join("outputs", os.path.basename(f)))
        z.write(os.path.join(REPO, "BUNDLE.json"), "BUNDLE.json")
    return f"{path} ({os.path.getsize(path) / 1e6:.1f} MB)"
print("ok")

ok


In [3]:
# 3. Run. Keep this cell running until it prints DONE.
from IPython.display import clear_output
if subprocess.run(["pgrep", "-f", "skyrag.cli eval"], capture_output=True, text=True).stdout.strip():
    raise SystemExit("Lanes from an earlier start are still running. Wait for them to finish, then run this cell again.")
notes = []
for prec in PRECISIONS:
    todo = [(ctx, group, [(lane_name(prec, ctx, cell, rep, h), arm, mode, qfile)
                          for cell, arm, mode in group_cells for rep in (1, 2, 3) for h, qfile in HALVES.items()])
            for ctx, group, group_cells in WAVES]
    pending = [n for _, _, lanes in todo for n, *_ in lanes if len(rows_of(n)) < ROWS]
    if pending:
        stop_any_server(); start_server(prec)
    for ctx, group, lanes in todo:
        procs = {}
        for name, arm, mode, qfile in lanes:
            if len(rows_of(name)) >= ROWS:
                continue
            shutil.rmtree(os.path.join(OUT, name), ignore_errors=True)     # a half-written lane is rerun whole
            procs[name] = launch(name, ["-Q", qfile, "-b", "openrouter", "-P", "skyrag", "--serving-precision", prec,
                                        "--sweep-version", "v5", "-m", SERVED, "--arm", arm, *CONTEXTS[ctx], "-r", "1",
                                        *(["--expansion-mode", mode] if mode else [])])
            time.sleep(5)
        t0 = time.time()
        while True:
            stats = [status(n) for n, *_ in lanes]
            gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader"],
                                 capture_output=True, text=True).stdout.strip()
            clear_output(wait=True)
            print("\n".join(notes + [f"{time.strftime('%H:%M:%S')}  Qwen {prec}, context {ctx}, {group} cells: "
                                     f"{sum(s[0] >= ROWS for s in stats)}/{len(lanes)} lanes done, "
                                     f"{(time.time() - t0) / 60:.0f} min   GPU: {gpu}"] + [s[1] for s in stats]), flush=True)
            if not any(p.poll() is None for p in procs.values()):
                break
            time.sleep(30)
        short = [n for n, *_ in lanes if len(rows_of(n)) < ROWS]
        notes.append(f"Qwen {prec}, context {ctx}, {group} cells: finished in {(time.time() - t0) / 60:.0f} min"
                     + (f"; INCOMPLETE lanes: {short}" if short else ""))
    if prec == "fp8":
        notes.append("fp8 zip ready: " + zip_run("qprec5090_fp8_results.zip", "*qprec5090_qwen_fp8*"))
stop_any_server()
notes.append("full zip ready: " + zip_run("qprec5090_results.zip", "*qprec5090_qwen_*"))
clear_output(wait=True)
print("\n".join(notes))
print("DONE. Download qprec5090_results.zip and send it to Claude.")

Qwen fp8, context s, core cells: finished in 0 min
Qwen fp8, context i, core cells: finished in 10 min
Qwen fp8, context s, decomposition cells: finished in 9 min
Qwen fp8, context i, decomposition cells: finished in 7 min
fp8 zip ready: /home/jovyan/work/TA-Sean-Matthew/qprec5090_fp8_results.zip (4.0 MB)
Qwen bf16, context s, core cells: finished in 15 min
Qwen bf16, context i, core cells: finished in 14 min
Qwen bf16, context s, decomposition cells: finished in 12 min
Qwen bf16, context i, decomposition cells: finished in 9 min
full zip ready: /home/jovyan/work/TA-Sean-Matthew/qprec5090_results.zip (8.1 MB)
DONE. Download qprec5090_results.zip and send it to Claude.
